# Financial Time Series I

Module: Financial Time Series

## Lesson summary

This notebook introduces financial time series through prices, returns, decomposition, autocorrelation, and the classical ARMA/ARIMA modeling vocabulary.

## Learning objectives
- Convert price data into time-indexed financial series.
- Describe time series components and decomposition.
- Interpret autocorrelation and partial autocorrelation plots.
- Distinguish white noise, random walks, AR, MA, ARMA, and ARIMA models.

## Lesson flow
1. Load and prepare a reproducible classroom price panel.
2. Create price and log-return series.
3. Study decomposition and correlation diagnostics.
4. Introduce classical time series models.


## Setup

In [ ]:
import pandas as pd
import numpy as np


# Visualization 
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Modeling
from statsmodels.tsa.seasonal import seasonal_decompose
from dateutil.parser import parse
from statsmodels.tsa.stattools import acf, pacf
# import statsmodels.formula.api as smf
import statsmodels.tsa.api as smt
import statsmodels.api as sm
import scipy.stats as scs
from statsmodels.tsa.ar_model import AutoReg, ar_select_order
from statsmodels.tsa.arima.model import ARIMA

from arch import arch_model


pd.set_option("display.max_columns",80)

In [ ]:
publication_end_date = "2026-06-05"
print("Publication data end date:", publication_end_date)

## Data

In [ ]:
start_date = "2018-01-01"
tickers = ['MAT','DIS','KO', 'NVDA','PFE', "AAPL", "META", "TSLA", "^GSPC","MSFT",]
print(f"The number of reproducible classroom price series is {len(tickers)}")

rng = np.random.default_rng(2026)
dates = pd.bdate_range(start=start_date, end=publication_end_date)
daily_mean = np.linspace(0.00010, 0.00045, len(tickers))
daily_volatility = np.linspace(0.009, 0.020, len(tickers))
log_returns = rng.normal(daily_mean, daily_volatility, size=(len(dates), len(tickers)))
adj_close = pd.DataFrame(
    100 * np.exp(np.cumsum(log_returns, axis=0)),
    index=dates,
    columns=tickers,
)
adj_close.index.name = "Date"
all_data = pd.concat({"Adj Close": adj_close}, axis=1)
all_data.info()

In [ ]:
adj_close_data_completed = all_data["Adj Close"].copy()
adj_close_data_completed.head()

## What is a time series?

A time series is a sequence of observations indexed by time. In financial mathematics, common examples include prices, returns, yields, exchange rates, volumes, and macroeconomic indicators. The time index matters because observations are usually dependent: today's value often contains information from previous values.

### Components of a time series

- **Trend:** persistent long-run movement in the level of the series.
- **Seasonality:** repeated behavior tied to a calendar frequency, such as month, quarter, or trading session.
- **Cycle:** lower-frequency movement that is not tied to a fixed calendar period.
- **Irregular component:** residual variation left after systematic structure has been modeled.
- **ETS decomposition:** a framework that separates error, trend, and seasonal components when that structure is appropriate {cite}`box2015time,tsay2010analysis`.

### Stationarity

<img src="../../img/generated/ts-stationarity-properties.png" alt="stationarity" style="height: 850px; width:650px;"/>

A stationary series has statistical behavior that is stable over time. In practice, this usually means focusing on a stable mean, variance, and autocovariance structure. Financial prices are often nonstationary, while transformed series such as returns or first differences are more suitable for many statistical models.

### Autocorrelation

Autocorrelation measures dependence between a series and lagged versions of itself. It is central to time-series modeling because it reveals whether past observations or past errors still contain usable structure.

Ignoring serial correlation can make model diagnostics misleading. Standard errors may be underestimated, test statistics may look stronger than they are, and residuals may still contain predictable information that the model failed to capture.

In [ ]:
apple_data = adj_close_data_completed['AAPL'].copy().reset_index().rename(
    columns={
        "AAPL":"value",
        "Date":"date"
    }
)
apple_data['date'] = pd.to_datetime(apple_data['date'].dt.date)

apple_data = apple_data[
    apple_data.date >= "2021-01-01"
].reset_index(
    drop =True
)

apple_data['log_returns'] = np.log(apple_data.value.pct_change() + 1)

    
apple_data.head()

In [ ]:
apple_data.set_index('date').value.plot(
    figsize=(12,6)
);
plt.title("Apple stock price");

In [ ]:
apple_data.set_index('date').log_returns.dropna().plot(
    figsize=(12,6)
);
plt.title("Apple log-returns");

## Price-to-volatility pipeline

Log returns convert a price path into a modeling series. The same return series is also the input for sample variance and annualized volatility estimates used later in risk and portfolio models.

![Pipeline from asset prices to log returns, variance, and annualized volatility](../../img/generated/ts-volatility-from-prices.png)


In [ ]:
apple_data.info()

In [ ]:
apple_data.date.dt.to_period('Y').value_counts()

## Decomposition of a Time Series 

In [ ]:
# Multiplicative Decomposition 
multiplicative_decomposition = seasonal_decompose(apple_data['value'], model='multiplicative', period=30)

# Plot
plt.rcParams.update({'figure.figsize': (16,12)})
multiplicative_decomposition.plot().suptitle('Multiplicative Decomposition', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

In [ ]:
# Additive Decomposition
additive_decomposition = seasonal_decompose(apple_data['value'], model='additive', period=30)

# Plot
additive_decomposition.plot().suptitle('Additive Decomposition', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

## Time series analysis

### Airline dataset

The airline passenger dataset shows the number of passengers per month from 1949 to 1960

In [ ]:
airpass = sm.datasets.get_rdataset("AirPassengers", "datasets")

In [ ]:
airpass_metadata = pd.Series(
    {
        "source": "R datasets package through statsmodels",
        "coverage": "Monthly observations from 1949-01 to 1960-12",
        "measure": "International airline passengers, in thousands",
        "classroom role": "Trend, seasonality, differencing, and decomposition",
    },
    name="value",
)
airpass_metadata.to_frame()

In [ ]:
airpass = pd.Series(airpass.data["value"])
airpass.index = pd.date_range(start="1949-01-31", periods=len(airpass.index), freq="ME").to_period("M")
airpass.index = airpass.index.to_timestamp()

airpass = airpass.reset_index()
airpass.columns = ['date', 'passengers']
airpass = airpass.set_index('date')

airpass.head()

In [ ]:
# Visualize
plt.title('Airline Passengers dataset', size=20)
plt.plot(airpass);

In [ ]:
# First-order difference
airpass['passengers_diff'] = airpass['passengers'].diff(periods=1)
airpass = airpass.dropna()

# Plot
plt.title('Airline Passengers dataset with First-order difference', size=16)
plt.plot(airpass['passengers'], label='Passengers')
plt.plot(airpass['passengers_diff'], label='First-order difference', color='orange')
plt.legend();

In [ ]:
airpass.head()

### Autocorrelation

Autocorrelation shows the correlation between a series and lagged versions of the same series. In an autocorrelation plot, the horizontal axis is the lag and the vertical axis is the correlation coefficient, which ranges from -1 to 1.

Autocorrelation is most interpretable after the modeling target has been made approximately stationary. For this passenger-count example, first differencing removes much of the trend and makes repeated seasonal dependence easier to see.

First differencing is not the only stationarity transformation, but it is a useful starting point for demonstrating lag dependence.

In [ ]:
airpass[["passengers", "passengers_diff"]].head()

#### Manual example

In [ ]:
lags = 4
# lags_aux = lags -1
df_lags = pd.concat(
    [
        airpass.passengers_diff,
        airpass.passengers_diff.shift(periods=lags).dropna(),
    ],
    axis=1
)

df_lags.columns = ['original',f'lag_{lags}']

display(df_lags.head(lags+5))

df_lags.corr().applymap("{:.2f}".format)

In [ ]:
acf_values = acf(airpass['passengers_diff'])
acf_values

In [ ]:
np.round(acf_values,2)

The first autocorrelation value is 1 because it compares the series with itself at lag 0. The lag-12 autocorrelation is high, which is consistent with annual seasonality in monthly passenger counts.

In [ ]:
plot_acf(airpass['passengers_diff'], lags=30);

The plot confirms strong seasonal dependence around lag 12. A related pattern appears around lag 24, although the relationship weakens as the lag grows.
The shaded area is the approximate confidence band. Autocorrelations inside that band should not be overinterpreted as statistically meaningful structure.

### PACF

*Partial autocorrelation*

Partial autocorrelation also compares a series with lagged versions of itself, but it controls for the shorter intervening lags. It is useful for separating direct lag effects from dependence that is transmitted through earlier lags.


The partial autocorrelation function (PACF) gives the partial correlation of a stationary time series with its own lagged values, regressed the values of the time series at all shorter lags. It contrasts with the autocorrelation function, which does not control for other lags.

This is the critical difference between autocorrelation and partial autocorrelation: ACF includes indirect lag dependence, while PACF isolates the incremental contribution of each lag.

In [ ]:
pacf_values = pacf(airpass['passengers_diff'])
np.round(pacf_values,2)

The next plot uses `statsmodels.graphics.tsaplots.plot_pacf` to estimate partial autocorrelations for the differenced passenger series. The lag count is kept small enough for the monthly sample size.

In [ ]:
plot_pacf(
    airpass['passengers_diff'], 
    lags=25,
    # method='yw'
    
);

### ACF and PACF plots

*How to interpret plots*

Autoregressive, moving-average, and ARMA models require lag-order choices. ACF and PACF plots provide diagnostic hints for those choices, although they should be combined with information criteria and residual checks.

Common heuristics:

* A gradually declining ACF with a sharply truncated PACF suggests an autoregressive structure.
* A sharply truncated ACF with a gradually declining PACF suggests a moving-average structure.
* Gradual decline in both plots suggests a mixed ARMA structure.
* No meaningful spikes in either plot suggests little linear dependence to model.

These are heuristics, not rules. A publishable workflow should compare candidate orders with AIC or BIC and then inspect residual autocorrelation.

## White noise

White noise is the benchmark residual process for many time-series models. A white-noise sequence has mean zero, no serial correlation, and constant variance. If a fitted model captures the systematic structure in a series, the remaining residuals should look close to white noise.

In [ ]:
def tsplot(y, lags=None, figsize=(10, 8), style='bmh'):
    if not isinstance(y, pd.Series):
        y = pd.Series(y)
    with plt.style.context(style):    
        fig = plt.figure(figsize=figsize)
        #mpl.rcParams['font.family'] = 'Ubuntu Mono'
        layout = (3, 2)
        ts_ax = plt.subplot2grid(layout, (0, 0), colspan=2)
        acf_ax = plt.subplot2grid(layout, (1, 0))
        pacf_ax = plt.subplot2grid(layout, (1, 1))
        qq_ax = plt.subplot2grid(layout, (2, 0))
        pp_ax = plt.subplot2grid(layout, (2, 1))
        
        y.plot(ax=ts_ax)
        ts_ax.set_title('Time Series Analysis Plots')
        smt.graphics.plot_acf(y, lags=lags, ax=acf_ax, alpha=0.5, )
        smt.graphics.plot_pacf(y, lags=lags, ax=pacf_ax, alpha=0.5, method='yw')
        sm.qqplot(y, line='s', ax=qq_ax)
        qq_ax.set_title('QQ Plot')        
        scs.probplot(y, sparams=(y.mean(), y.std()), plot=pp_ax)

        plt.tight_layout()
    return 

In [ ]:
np.random.seed(1)

# plot of discrete white noise
randser = np.random.normal(size=10000)
tsplot(randser, lags=30)

The simulated process is centered near zero and has no persistent serial structure. A small number of apparently significant ACF or PACF spikes can occur by chance. The QQ and probability plots compare the simulated distribution with a Gaussian reference distribution.

In [ ]:
print(
f"""
Random Series
--------------
mean: {randser.mean():.3f}
variance: {randser.var():.3f}
standard deviation: {randser.std():.3f}
"""
)

## Random walk

A random walk is a process in which each new value equals the previous value plus a random innovation:

$$
x_t = x_{t-1} + w_t.
$$

Random walks are important in finance because price levels often behave more like accumulated shocks than mean-reverting stationary series. The level is nonstationary, but the first difference can be stationary if the innovations are white noise.

The next cell simulates a random walk.

In [ ]:
np.random.seed(1)
n_samples = 1000

x = w = np.random.normal(size=n_samples)
for t in range(n_samples):
    x[t] = x[t-1] + w[t]

_ = tsplot(x, lags=25)

The simulated level is not stationary. For a random walk, differencing gives $x_t - x_{t-1} = w_t$, so the first difference should behave like white noise when the random-walk assumption is correct.

In [ ]:
_ = tsplot(np.diff(x), lags=20)

Our definition holds as this looks exactly like a white noise process

In [ ]:
_ = tsplot(np.diff(apple_data.value), lags=40)

The differenced price series is much closer to white noise than the original level series. The QQ and probability plots still show sample tail variation, and a few ACF/PACF spikes can appear by chance. The key diagnostic point is that differencing removes the persistent nonstationary structure from the level series.

## Linear Models

A deterministic linear trend model represents a time series as a straight-line function of time plus an error term. The basic equation is:


$$
y_{t} = \beta_{0} + \beta_{1}t + \epsilon_{t}
$$

Here, time is the explanatory variable. This model can describe a trend, but its residuals still need to be checked for autocorrelation and changing variance.


In [ ]:
w = np.random.randn(500)
y = np.empty_like(w)

b0 = -50.
b1 = 25.
for t in range(len(w)):
    y[t] = b0 + b1*t + w[t]
    
_ = tsplot(y, lags=50)  

The residual diagnostics show a clear autocorrelation pattern that declines with the lag. The marginal distribution is approximately normal, but the residual dependence indicates that a simple deterministic trend is not enough. The PACF spike at lag 1 suggests that an autoregressive model may be appropriate.

## Log-Linear Models

Log-linear models are similar to linear trend models, but the level follows an exponential pattern. This structure is appropriate when the series grows by an approximately constant percentage rate each period. The simulated example below shows the curved level path and its linearized log transformation.

In [ ]:
# Simulate deterministic exponential growth.
idx = pd.date_range("2007-01-31", "2011-12-31", freq="ME")
sales = [np.exp(x / 12) for x in range(1, len(idx) + 1)]
df = pd.DataFrame(sales, columns=["Sales"], index=idx)

with plt.style.context("bmh"):
    df.plot()
    plt.title("Simulated Exponential Growth")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

We can then transform the data by taking the natural logarithm of sales. Now a linear regression is a much better fit to the data.


In [ ]:
# ABC log sales 

with plt.style.context('bmh'):
    pd.Series(np.log(sales), index=idx).plot()
    plt.title('ABC Log Sales')

Deterministic trend models are useful baselines, but they are often incomplete for financial time series. Residual autocorrelation indicates that the model has not captured all time dependence, which motivates autoregressive specifications.

## AR(p)

*Autoregressive Models*

An autoregressive model explains the current value of a series using one or more of its own lagged values:

$$
x_{t} = \alpha_{1}x_{t-1} + ... +  \alpha_{p}x_{t-p} + \omega_{t}
$$

The order $p$ is the number of lagged values included in the model. For example, an AR(2) model is:

$$
x_{t} = \alpha_{1}x_{t-1} + \alpha_{2}x_{t-2} + \omega_{t}
$$

Here, $\alpha$ coefficients measure lag dependence and $\omega_t$ is a white-noise innovation. An AR(1) model with $\alpha = 1$ is a random walk and is therefore nonstationary.

$$
x_{t} = 1x_{t-1} + \omega_{t}
$$

The next cell simulates an AR(1) process with $\alpha = 0.6$.

In [ ]:
# Simulate an AR(1) process with alpha = 0.6

np.random.seed(1)
n_samples = int(1000)
a = 0.6
x = w = np.random.normal(size=n_samples)

for t in range(n_samples):
    x[t] = a*x[t-1] + w[t]
    
_ = tsplot(x, lags=25)

As expected the distribution of our simulated AR(1) model is normal. There is significant serial correlation between lagged values especially at lag 1 as evidenced by the PACF plot. 

The simulated series can now be fit with `statsmodels`. The first fit estimates the AR coefficient. The order-selection step then checks whether the model identifies lag 1 as the preferred autoregressive order. For this controlled simulation, the estimated coefficient should be close to the true value $\alpha = 0.6$.

In [ ]:
# Fit an AR(p) model to simulated AR(1) model with alpha = 0.6

mdl = AutoReg(x, lags=1, trend='t').fit()
est_order = ar_select_order(
    x,
    maxlag=30, ic='aic', trend='t'
)

true_order = 1
print(
f'''
alpha estimate: {mdl.params[1]:3.5f} | best lag order = {est_order.ar_lags}
'''
)

print(
f'''
true alpha: {a:3.5f} | true order = {true_order}
'''
)

# print('\ntrue alpha = {} | true order = {}'
#   .format(a, true_order))

The fitted model recovers the main dependence pattern from the simulated data. The next example simulates an AR(2) process with $\alpha_1 = 0.666$ and $\alpha_2 = -0.333$ using `statsmodels.tsa.arima_process.arma_generate_sample`.

In [ ]:
# Simulate an AR(2) process

n = int(1000)
alphas = np.array([.666, -.333])
betas = np.array([0.])

# Python requires us to specify the zero-lag value which is 1
# Also note that the alphas for the AR model must be negated
# We also set the betas for the MA equal to 0 for an AR(p) model
# For more information see the examples at statsmodels.org
ar = np.r_[1, -alphas]
ma = np.r_[1, betas]

ar2 = smt.arma_generate_sample(ar=ar, ma=ma, nsample=n) 
_ = tsplot(ar2, lags=30)

In [ ]:
mdl = AutoReg(ar2, lags=2, trend='t').fit()
est_order = ar_select_order(
    ar2,
    maxlag=30, ic='aic', trend='t'
)

true_order = 2
print(
f'''
alpha estimate: {mdl.params[1:]:} | best lags order = {est_order.ar_lags}
'''
)

print(
f'''
true alpha: {[.666, -.333]} | true order = {true_order}
'''
)

In [ ]:
pip freeze | grep stat

### Apple data

In [ ]:
_ = tsplot(apple_data.log_returns.dropna() , lags=30)

In [ ]:
est_order = ar_select_order(
    apple_data.log_returns.dropna().values,
    maxlag=13, ic='aic', trend='t', glob=True
)

In [ ]:
est_order.ar_lags

## MA(q)

*Moving Average Models*

A moving-average model represents the current value of a series as a linear combination of current and past white-noise innovations.

The difference from AR(p) is the source of dependence. AR models use past observations; MA models use past shocks. The formula for an MA(q) model is:

$$
x_{t} = \omega_{t} + \beta_{1}\omega_{t-1} + ... +  \beta_{q}\omega_{t-q} = \omega_{t} + \sum_{i=1}^{Q} \beta_{i}\omega_{t-i}
$$

$\omega_t$ is white noise with $E(\omega_t)=0$ and variance $\sigma^2$. The next cell simulates an MA(1) process with $\beta=0.6$.

In [ ]:
# Simulate an MA(1) process

n = int(1000)

# set the AR(p) alphas equal to 0
alphas = np.array([0.])
betas = np.array([0.6])

# add zero-lag and negate alphas
ar = np.r_[1, -alphas]
ma = np.r_[1, betas]

ma1 = smt.arma_generate_sample(ar=ar, ma=ma, nsample=n) 
_ = tsplot(ma1, lags=30)

The ACF function shows that lag 1 is significant which indicates that a MA(1) model may be appropriate for our simulated series.

In [ ]:
mdl = ARIMA(
    ma1,
    order=(0, 0, 1),
    trend = "n",
    seasonal_order = (0,0,0,0)
).fit()
mdl.summary()

The model was able to correctly estimate the lag coefficent as 0.58 is close to our true value of 0.6. Also notice that our 95% confidence interval does contain the true value.

## ARMA(p, q)

*Autoregressive Moving Average Models*

The ARMA model is simply the merger between AR(p) and MA(q) models.

AR(p) models try to capture (explain) the momentum and mean reversion effects often observed in trading markets.

MA(q) models try to capture (explain) the shock effects observed in the white noise terms. These shock effects could be thought of as unexpected events affecting the observation process

ARMA's weakness is that it ignores the volatility clustering effects found in most financial time series. 

The model formula is:

$$
x_{t} = \alpha_{1}x_{t-1} + \alpha_{2}x_{t-2} + ... +  \omega_{t} + \beta_{1}\omega_{t-1} + ... +  \beta_{q}\omega_{t-q}
$$


$$
= \sum_{i=1}^{P} \alpha_{i}x_{t-i} + \omega_{t} + \sum_{i=1}^{Q} \beta_{i}\omega_{t-i}
$$


In [ ]:
max_lag = 30

n = int(5000) # lots of samples to help estimates
burn = int(n/10) # number of samples to discard before fit

alphas = np.array([0.5, -0.25])
betas = np.array([0.5, -0.3])
ar = np.r_[1, -alphas]
ma = np.r_[1, betas]

arma22 = smt.arma_generate_sample(ar=ar, ma=ma, nsample=n, burnin=burn)
_ = tsplot(arma22, lags=max_lag)

In [ ]:
mdl = ARIMA(
    arma22,
    order=(2, 0, 2),
    trend = "n",
    seasonal_order = (0,0,0,0)
).fit()
mdl.summary()

The model has correctly recovered our parameters, and our true parameters are contained within the 95% confidence interval.

## ARIMA(p, d, q)

*Autoregressive Integrated Moving Average Models*

ARIMA extends ARMA by adding differencing. Many time series are not stationary in levels, but can be transformed into a more stable modeling target by differencing. The random-walk example showed the core idea: first differencing a nonstationary random walk recovers the innovation process.

In [ ]:
best_aic = np.inf 
best_order = None
best_mdl = None

pq_rng = range(5) # [0,1,2,3,4]
d_rng = range(2) # [0,1]
for i in pq_rng:
    for d in d_rng:
        for j in pq_rng:
            try:
                # tmp_mdl = ARIMA(lrets.SPY, order=(i,d,j)).fit(method='mle', trend='nc')
                tmp_mdl = ARIMA(
                    apple_data.log_returns.dropna().values,
                    order=(i,d,j),
                    trend = "n",
                    seasonal_order = (0,0,0,0)
                ).fit()
                tmp_aic = tmp_mdl.aic
                if tmp_aic < best_aic:
                    best_aic = tmp_aic
                    best_order = (i, d, j)
                    best_mdl = tmp_mdl
            except: continue


print('aic: {:6.5f} | order: {}'.format(best_aic, best_order))

# ARIMA model resid plot
_ = tsplot(best_mdl.resid, lags=30)

## Further references

For deeper treatment of ARIMA and classical time-series modeling, see {cite}`box2015time,hamilton1994time,tsay2010analysis`. For volatility models and Python implementation details used later in the module, see {cite}`engle1982autoregressive,bollerslev1986generalized,sheppard2024arch,statsmodels2010`.